# Датасет по модели численности (resist_id=39)

Цель: собрать датафрейм из CSV (raw) со столбцами:
- id
- month
- year
- operation_id
- fact_value
- fact_predict
- predict_new (пока пустое)
- workDayMonth_2


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAW_DIR = Path("../data/raw")
RAW_FILES = [
    RAW_DIR / "data-2022.csv",
    RAW_DIR / "data-2023.csv",
    RAW_DIR / "data-2024.csv",
    RAW_DIR / "data-2025.csv",
]

RAW_FILES

In [ ]:
# Загружаем все CSV (все модели численности)

dfs = []
for p in RAW_FILES:
    df_part = pd.read_csv(p, dtype=str)
    dfs.append(df_part)

df = pd.concat(dfs, ignore_index=True)

df["period"] = pd.to_datetime(df["period"], errors="coerce")
df["resist_id"] = pd.to_numeric(df["resist_id"], errors="coerce").astype("Int64")

# работаем по всем resist_id

df_all = df.copy()

print("Всего строк (все resist_id):", len(df_all))
df_all.head()

In [ ]:
# Data Cleaning: фильтруем только утвержденные записи до парсинга JSON

# Нормализация булевых полей

def to_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def to_bool_value(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return None
    return str(value).strip().lower() in ["true", "t", "1"]


bool_cols = ["send", "validate_one", "validate_two", "cancel_approve", "forecast"]
for col in bool_cols:
    if col in df_all.columns:
        df_all[col] = to_bool(df_all[col])

# Валидация этапов

df_all["validate_count"] = pd.to_numeric(df_all["validate_count"], errors="coerce").fillna(0).astype(int)

mask_sent = df_all["send"]
mask_not_cancelled = ~df_all["cancel_approve"]
mask_approved = (
    (df_all["validate_count"] == 0)
    | ((df_all["validate_count"] == 1) & df_all["validate_one"])
    | ((df_all["validate_count"] >= 2) & df_all["validate_one"] & df_all["validate_two"])
)

# Исключение авто-прогнозов (Data Leakage / Loop prevention)

PREDICT_COMMENT = "first_predict_value_record"
if "comment" in df_all.columns:
    df_all["comment"] = df_all["comment"].fillna("")
    mask_predict_record = df_all["comment"] == PREDICT_COMMENT
else:
    mask_predict_record = pd.Series(False, index=df_all.index)

# Итоговый фильтр (важно: forecast не фильтруем!)

df_ok = df_all[mask_sent & mask_not_cancelled & mask_approved & ~mask_predict_record].copy()

# Разбор operation_parameter + calculate_parameter

def parse_division_title(div_json: str):
    if pd.isna(div_json):
        return None
    try:
        obj = json.loads(div_json)
    except Exception:
        return None
    return obj.get("name")


def parse_calculate_param(calc_json: str):
    if pd.isna(calc_json):
        return {}
    try:
        return json.loads(calc_json)
    except Exception:
        return {}


def get_calc_value(calc_obj: dict, key: str):
    val = calc_obj.get(key)
    if isinstance(val, dict):
        return val.get("value")
    return val


records = []
for _, row in df_ok.iterrows():
    calc_obj = parse_calculate_param(row.get("calculate_parameter"))
    workday = get_calc_value(calc_obj, "workDayMonth_2")
    op_raw = row.get("operation_parameter")
    if pd.isna(op_raw):
        continue
    try:
        op_data = json.loads(op_raw)
    except Exception:
        continue

    for op_id, payload in op_data.items():
        if not isinstance(payload, dict):
            continue
        # исключаем неактивные операции
        active_flag = to_bool_value(payload.get("operation_active", payload.get("active")))
        if active_flag is False:
            continue

        period = row.get("period")
        if pd.isna(period):
            continue
        records.append({
            "resist_id": row.get("resist_id"),
            "division_title": parse_division_title(row.get("division_name")),
            "id": row.get("id"),
            "month": int(period.month),
            "year": int(period.year),
            "operation_id": op_id,
            "fact_value": payload.get("fact_value"),
            "fact_predict": payload.get("fact_predict"),
            "predict_catboost_ver1": None,
            "mape_catboost_ver1": pd.NA,
            "workDayMonth_2": workday,
            "calculate_minute": payload.get("calculate_minute"),
        })

df_model39 = pd.DataFrame(records)

# Типы и базовые поля

df_model39["fact_value"] = pd.to_numeric(df_model39["fact_value"], errors="coerce")
df_model39["fact_predict"] = pd.to_numeric(df_model39["fact_predict"], errors="coerce")
df_model39["workDayMonth_2"] = pd.to_numeric(df_model39["workDayMonth_2"], errors="coerce")
df_model39["calculate_minute"] = pd.to_numeric(df_model39["calculate_minute"], errors="coerce")

# коэффициент полной занятости

df_model39["coefficient_fulltime_staff"] = (
    df_model39["calculate_minute"] / 60 / 148.3
)

# значения в FTE (fact vs predict)

df_model39["fact_fts"] = df_model39["fact_value"] * df_model39["coefficient_fulltime_staff"]
df_model39["predict_fts"] = df_model39["fact_predict"] * df_model39["coefficient_fulltime_staff"]

# MAPE по текущему прогнозу (fact_predict vs fact_value)

df_model39["mape_fact_predict"] = (
    (df_model39["fact_value"] - df_model39["fact_predict"]).abs()
    / df_model39["fact_value"].abs()
) * 100

# если появились бесконечности — заменяем на NA

df_model39["mape_fact_predict"] = df_model39["mape_fact_predict"].replace(
    [float("inf"), float("-inf")], pd.NA
)

# Правила для fact_value == 0:
# - если fact_predict == 0 => MAPE = 0 (0%)
# - если fact_predict != 0 => MAPE = 1 (100%)
mask_zero = df_model39["fact_value"] == 0
mask_zero_pred = mask_zero & (df_model39["fact_predict"] == 0)
mask_zero_nonzero_pred = mask_zero & (df_model39["fact_predict"] != 0)

df_model39.loc[mask_zero_pred, "mape_fact_predict"] = 0.0
df_model39.loc[mask_zero_nonzero_pred, "mape_fact_predict"] = 100.0

# large_error_share для текущего прогноза

df_model39["large_error_share"] = df_model39["mape_fact_predict"] > 30

# квадрат ошибки (в FTE)

df_model39["square_error_fact_predict"] = (
    (df_model39["fact_value"] * df_model39["coefficient_fulltime_staff"])
    - (df_model39["fact_predict"] * df_model39["coefficient_fulltime_staff"])
).abs() ** 2

# Feature Engineering (Time Series)

# дата из года и месяца

df_model39["date"] = pd.to_datetime(
    dict(year=df_model39["year"], month=df_model39["month"], day=1),
    errors="coerce",
).dt.date

# сортировка обязательна для лагов

df_model39 = df_model39.sort_values(["operation_id", "year", "month"]).reset_index(drop=True)

# лаги и скользящие средние (по operation_id)

group = df_model39.groupby("operation_id")

df_model39["lag_1"] = group["fact_value"].shift(1)

df_model39["lag_12"] = group["fact_value"].shift(12)

df_model39["roll_mean_3"] = group["fact_value"].transform(
    lambda s: s.shift(1).rolling(3, min_periods=3).mean()
)

df_model39["roll_mean_6"] = group["fact_value"].transform(
    lambda s: s.shift(1).rolling(6, min_periods=6).mean()
)

# тренд: отношение прошлого месяца к среднему за 3 мес

df_model39["trend"] = df_model39["lag_1"] / df_model39["roll_mean_3"]

# убираем бесконечности, которые могли появиться в тренде

df_model39["trend"] = df_model39["trend"].replace([np.inf, -np.inf], 0)

# Финальная очистка

# 1) удаляем строки без факта

df_model39 = df_model39.dropna(subset=["fact_value"])

# 2) удаляем первые 12 месяцев (лаг 12)

df_model39 = df_model39.dropna(subset=["lag_12"])

# 3) оставшиеся пропуски заполняем нулями (только числовые поля)

num_cols = df_model39.select_dtypes(include=["number"]).columns

df_model39[num_cols] = df_model39[num_cols].fillna(0)

# приведение типов после fillna (для future-совместимости pandas)

df_model39 = df_model39.infer_objects(copy=False)

# порядок колонок

ordered_cols = [
    "resist_id",
    "division_title",
    "date",
    "operation_id",
    "fact_value",
    "fact_predict",
    "fact_fts",
    "predict_fts",
    "mape_fact_predict",
    "large_error_share",
    "square_error_fact_predict",
    "predict_catboost_ver1",
    "mape_catboost_ver1",
    "id",
    "month",
    "year",
    "calculate_minute",
    "coefficient_fulltime_staff",
    "lag_1",
    "lag_12",
    "roll_mean_3",
    "roll_mean_6",
    "trend",
]

extra_cols = [c for c in df_model39.columns if c not in ordered_cols]

if extra_cols:
    df_model39 = df_model39[ordered_cols + extra_cols]
else:
    df_model39 = df_model39[ordered_cols]

print("Итоговых строк:", len(df_model39))
df_model39.head()

In [ ]:
# Диагностика: где отсекаются месяцы
# Показываем число строк по месяцам и последний доступный месяц на каждом этапе.

def _month_stats_from_period(df: pd.DataFrame, stage_name: str):
    if df is None or df.empty:
        print(f"[{stage_name}] пусто")
        return

    if "period" not in df.columns:
        print(f"[{stage_name}] нет колонки period")
        return

    s = pd.to_datetime(df["period"], errors="coerce")
    s = s.dropna()
    if s.empty:
        print(f"[{stage_name}] period пуст после to_datetime")
        return

    month_series = s.dt.to_period("M").astype(str)
    counts = month_series.value_counts().sort_index()

    print(f"\n[{stage_name}] строк: {len(df)}")
    print("Последний месяц в массиве данных:", month_series.max())
    print("Распределение по месяцам:")
    print(counts.to_string())


def _month_stats_from_year_month(df: pd.DataFrame, stage_name: str):
    if df is None or df.empty:
        print(f"[{stage_name}] пусто")
        return

    required = {"year", "month"}
    if not required.issubset(df.columns):
        print(f"[{stage_name}] нет year/month")
        return

    ym = pd.to_datetime(
        dict(
            year=pd.to_numeric(df["year"], errors="coerce"),
            month=pd.to_numeric(df["month"], errors="coerce"),
            day=1,
        ),
        errors="coerce",
    ).dropna()

    if ym.empty:
        print(f"[{stage_name}] year/month пусты после to_datetime")
        return

    month_series = ym.dt.to_period("M").astype(str)
    counts = month_series.value_counts().sort_index()

    print(f"\n[{stage_name}] строк: {len(df)}")
    print("Последний месяц в массиве данных:", month_series.max())
    print("Распределение по месяцам:")
    print(counts.to_string())


print("=== Диагностика отсечения по месяцам ===")
_month_stats_from_period(df_all, "df_all (до фильтра)")
_month_stats_from_period(df_ok, "df_ok (после фильтра)")
_month_stats_from_year_month(df_model39, "df_model39 (после парсинга и очистки)")

if not df_model39.empty:
    last_fact_month = pd.to_datetime(
        dict(year=df_model39["year"], month=df_model39["month"], day=1),
        errors="coerce",
    ).max()
    print("\nПоследний месяц, данные за который есть в итоговом массиве:", last_fact_month.strftime("%Y-%m") if pd.notna(last_fact_month) else "нет данных")

In [ ]:
# Расширенная диагностика фильтров: помесячные маски и последовательный отсев

def _to_bool_diag(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "t", "1"])


def _last_month_from_period_series(period_series: pd.Series):
    s = pd.to_datetime(period_series, errors="coerce").dropna()
    if s.empty:
        return None
    return s.dt.to_period("M").max()


if df_all.empty:
    print("df_all пуст")
else:
    diag = df_all.copy()
    diag["period_dt"] = pd.to_datetime(diag.get("period"), errors="coerce")
    diag = diag[diag["period_dt"].notna()].copy()

    if diag.empty:
        print("В df_all нет валидных period")
    else:
        diag["month_key"] = diag["period_dt"].dt.to_period("M").astype(str)

        # Нормализация колонок для масок
        for c in ["send", "cancel_approve", "validate_one", "validate_two", "forecast"]:
            if c not in diag.columns:
                diag[c] = False
            diag[c] = _to_bool_diag(diag[c])

        if "comment" not in diag.columns:
            diag["comment"] = ""
        diag["comment"] = diag["comment"].fillna("")

        diag["validate_count"] = pd.to_numeric(diag.get("validate_count"), errors="coerce").fillna(0).astype(int)

        # Маски (как в основном коде)
        m_send = diag["send"]
        m_not_cancel = ~diag["cancel_approve"]
        m_approved = (
            (diag["validate_count"] == 0)
            | ((diag["validate_count"] == 1) & diag["validate_one"])
            | ((diag["validate_count"] >= 2) & diag["validate_one"] & diag["validate_two"])
        )
        m_not_predict = diag["comment"] != "first_predict_value_record"

        m_step1 = m_send
        m_step2 = m_step1 & m_not_cancel
        m_step3 = m_step2 & m_approved
        m_step4 = m_step3 & m_not_predict  # итоговый df_ok

        # Помесячная таблица
        monthly = (
            diag.groupby("month_key", as_index=False)
            .agg(
                total_rows=("month_key", "size"),
                send_true=("send", "sum"),
                cancel_approve_true=("cancel_approve", "sum"),
                not_cancelled_true=("cancel_approve", lambda s: (~s).sum()),
                approved_true=("validate_count", lambda s: m_approved.loc[s.index].sum()),
                predict_comment_true=("comment", lambda s: (s == "first_predict_value_record").sum()),
                step1_send=("validate_count", lambda s: m_step1.loc[s.index].sum()),
                step2_send_not_cancel=("validate_count", lambda s: m_step2.loc[s.index].sum()),
                step3_plus_validate=("validate_count", lambda s: m_step3.loc[s.index].sum()),
                step4_final_df_ok=("validate_count", lambda s: m_step4.loc[s.index].sum()),
            )
            .sort_values("month_key")
            .reset_index(drop=True)
        )

        print("=== Расширенная диагностика масок по месяцам ===")
        print(monthly.to_string(index=False))

        def _print_last_month(mask: pd.Series, label: str):
            if mask.any():
                last_m = diag.loc[mask, "month_key"].max()
                print(f"Последний месяц для {label}: {last_m}")
            else:
                print(f"Последний месяц для {label}: нет данных")

        print("\n=== Последний месяц по шагам фильтра ===")
        _print_last_month(pd.Series(True, index=diag.index), "df_all (валидный period)")
        _print_last_month(m_step1, "step1 send")
        _print_last_month(m_step2, "step2 send & ~cancel_approve")
        _print_last_month(m_step3, "step3 + validation")
        _print_last_month(m_step4, "step4 + comment != first_predict_value_record (df_ok)")

        # Сверка с уже построенными наборами
        last_df_ok = _last_month_from_period_series(df_ok.get("period")) if "period" in df_ok.columns else None
        print("Последний месяц в df_ok (фактический набор):", str(last_df_ok) if last_df_ok is not None else "нет данных")

        if not df_model39.empty and {"year", "month"}.issubset(df_model39.columns):
            dt_model = pd.to_datetime(
                dict(
                    year=pd.to_numeric(df_model39["year"], errors="coerce"),
                    month=pd.to_numeric(df_model39["month"], errors="coerce"),
                    day=1,
                ),
                errors="coerce",
            ).dropna()
            if not dt_model.empty:
                print("Последний месяц в df_model39 (после очистки):", str(dt_model.dt.to_period("M").max()))
            else:
                print("Последний месяц в df_model39 (после очистки): нет данных")
        else:
            print("Последний месяц в df_model39 (после очистки): нет данных")

In [ ]:
# Прицельная диагностика Q4 2025: почему не проходят месяцы 2025-10..2025-12
# Использует объекты из расширенной диагностики: diag, m_send, m_not_cancel, m_approved, m_not_predict.

q4_months = ["2025-10", "2025-11", "2025-12"]

if "diag" not in globals():
    print("Сначала запустите ячейку 'Расширенная диагностика фильтров'.")
else:
    diag_q4 = diag[diag["month_key"].isin(q4_months)].copy()

    if diag_q4.empty:
        print("В исходном массиве нет строк за 2025-10..2025-12")
    else:
        diag_q4["m_send"] = m_send.loc[diag_q4.index]
        diag_q4["m_not_cancel"] = m_not_cancel.loc[diag_q4.index]
        diag_q4["m_approved"] = m_approved.loc[diag_q4.index]
        diag_q4["m_not_predict"] = m_not_predict.loc[diag_q4.index]

        diag_q4["step1"] = diag_q4["m_send"]
        diag_q4["step2"] = diag_q4["step1"] & diag_q4["m_not_cancel"]
        diag_q4["step3"] = diag_q4["step2"] & diag_q4["m_approved"]
        diag_q4["step4"] = diag_q4["step3"] & diag_q4["m_not_predict"]

        summary_q4 = (
            diag_q4.groupby("month_key", as_index=False)
            .agg(
                total_rows=("month_key", "size"),
                send_true=("m_send", "sum"),
                not_cancel_true=("m_not_cancel", "sum"),
                approved_true=("m_approved", "sum"),
                not_predict_true=("m_not_predict", "sum"),
                step1_pass=("step1", "sum"),
                step2_pass=("step2", "sum"),
                step3_pass=("step3", "sum"),
                step4_pass=("step4", "sum"),
                validate_count_0=("validate_count", lambda s: (s == 0).sum()),
                validate_count_1=("validate_count", lambda s: (s == 1).sum()),
                validate_count_ge2=("validate_count", lambda s: (s >= 2).sum()),
                v1_true=("validate_one", "sum"),
                v2_true=("validate_two", "sum"),
                predict_comment_true=("comment", lambda s: (s == "first_predict_value_record").sum()),
                cancel_approve_true=("cancel_approve", "sum"),
            )
            .sort_values("month_key")
            .reset_index(drop=True)
        )

        print("=== Q4 2025: breakdown по маскам ===")
        print(summary_q4.to_string(index=False))

        print("\n=== Топ причин отсечения в Q4 2025 (по строкам) ===")
        reason_rows = []
        for m in q4_months:
            sub = diag_q4[diag_q4["month_key"] == m]
            if sub.empty:
                continue
            reason_rows.append({
                "month_key": m,
                "total": len(sub),
                "fail_send": int((~sub["m_send"]).sum()),
                "fail_cancel": int((sub["m_send"] & ~sub["m_not_cancel"]).sum()),
                "fail_validation": int((sub["m_send"] & sub["m_not_cancel"] & ~sub["m_approved"]).sum()),
                "fail_comment": int((sub["m_send"] & sub["m_not_cancel"] & sub["m_approved"] & ~sub["m_not_predict"]).sum()),
                "final_pass": int(sub["step4"].sum()),
            })

        if reason_rows:
            print(pd.DataFrame(reason_rows).to_string(index=False))

        # Показываем небольшой сэмпл строк, которые не прошли step3 (валидацию)
        failed_step3 = diag_q4[diag_q4["step2"] & ~diag_q4["step3"]].copy()
        if not failed_step3.empty:
            show_cols = [
                c for c in [
                    "id", "period", "month_key", "send", "cancel_approve",
                    "validate_count", "validate_one", "validate_two", "comment", "forecast"
                ] if c in failed_step3.columns
            ]
            print("\n=== Пример строк Q4, не прошедших step3 (валидацию) ===")
            print(failed_step3[show_cols].head(20).to_string(index=False))

## Обучение CatBoost: какие признаки используем

Категориальные признаки (CatBoost их обрабатывает напрямую):
- `operation_id` — критически важно
- `division_title` — помогает учитывать специфику отдела
- `resist_id` — идентификатор модели численности

Числовые признаки (Time Series):
- `month`, `year` — сезонность
- `workDayMonth_2` — внешний регрессор (рабочие дни)
- `lag_1`, `lag_12` — точечные лаги
- `roll_mean_3`, `roll_mean_6` — скользящие средние
- `trend` — отношение `lag_1 / roll_mean_3`

Дополнительные признаки:
- `calculate_minute`, `coefficient_fulltime_staff`

Цель (target): `fact_value` — фактическое количество операций.

Идентификаторы `resist_id`, `id`, `date` в модель не подаются.

In [ ]:
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# признаки и цель
cat_cols = ["operation_id", "division_title", "resist_id"]
num_cols = [
    "month",
    "year",
    "workDayMonth_2",
    "calculate_minute",
    "coefficient_fulltime_staff",
    "lag_1",
    "lag_12",
    "roll_mean_3",
    "roll_mean_6",
    "trend",
]

feature_cols = cat_cols + num_cols

target_col = "fact_value"

# простое разделение train/test по времени (последний год — тест)

df_model39 = df_model39.sort_values(["year", "month"]).copy()

# CatBoost ожидает категориальные признаки в виде строк

df_model39[cat_cols] = df_model39[cat_cols].astype(str)

last_year = df_model39["year"].max()
train_df = df_model39[df_model39["year"] < last_year]
test_df = df_model39[df_model39["year"] == last_year]

X_train = train_df[feature_cols]
y_train = train_df[target_col]
X_test = test_df[feature_cols]
y_test = test_df[target_col]

cat_feature_indices = [feature_cols.index(c) for c in cat_cols]

model = CatBoostRegressor(
    iterations=1000,
    depth=6,
    learning_rate=0.1,
    loss_function="RMSE",
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
)

# Используем тест (2025) как eval_set для early stopping
# Веса: чем больше норматив времени, тем важнее строка
train_weight = train_df["calculate_minute"].astype(float)
train_weight = train_weight.fillna(train_weight.mean())

model.fit(
    X_train,
    y_train,
    cat_features=cat_feature_indices,
    eval_set=(X_test, y_test),
    sample_weight=train_weight,
)

pred = model.predict(X_test)

# Пост-обработка: отсечка отрицательных прогнозов
pred = np.maximum(pred, 0)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

# MAPE для CatBoost (pred vs fact)
mape_model = (y_test - pred).abs() / y_test.abs()
# правила для fact_value == 0
mask_zero = y_test == 0
mape_model[mask_zero & (pred == 0)] = 0.0
mape_model[mask_zero & (pred != 0)] = 1.0
mape_model = mape_model.replace([float("inf"), float("-inf")], pd.NA)
mape_model = mape_model.dropna().mean() * 100

# MAPE для текущего прогноза fact_predict на той же тестовой выборке
fact_pred_test = test_df["fact_predict"].astype(float)
mape_fact_pred = (y_test - fact_pred_test).abs() / y_test.abs()
mape_fact_pred[mask_zero & (fact_pred_test == 0)] = 0.0
mape_fact_pred[mask_zero & (fact_pred_test != 0)] = 1.0
mape_fact_pred = mape_fact_pred.replace([float("inf"), float("-inf")], pd.NA)
mape_fact_pred = mape_fact_pred.dropna().mean() * 100

print(f"Test MAE: {mae:.2f}")
print(f"Test RMSE: {rmse:.2f}")
print(f"Test MAPE (CatBoost): {mape_model:.2f}%")
print(f"Test MAPE (fact_predict): {mape_fact_pred:.2f}%")

# Сохраняем прогноз в Excel (тестовый год)

df_pred = test_df.copy()
df_pred["predict_catboost_ver1"] = pred

# MAPE для CatBoost по строкам теста
mape_cat = (df_pred["fact_value"] - df_pred["predict_catboost_ver1"]).abs() / df_pred["fact_value"].abs()
mask_zero_pred = df_pred["fact_value"] == 0
mape_cat[mask_zero_pred & (df_pred["predict_catboost_ver1"] == 0)] = 0.0
mape_cat[mask_zero_pred & (df_pred["predict_catboost_ver1"] != 0)] = 1.0
mape_cat = mape_cat.replace([float("inf"), float("-inf")], pd.NA)

# сохраняем MAPE в процентах
mape_cat = mape_cat * 100

df_pred["mape_catboost_ver1"] = mape_cat

# large_error_share для CatBoost (на уровне строки)
df_pred["large_error_share_catboost_ver1"] = df_pred["mape_catboost_ver1"] > 30

# square_error для CatBoost (в FTE)
coef = df_pred["coefficient_fulltime_staff"]
df_pred["square_error_catboost_ver1"] = (
    (df_pred["fact_value"] * coef) - (df_pred["predict_catboost_ver1"] * coef)
).abs() ** 2

out_xlsx = "../data/processed/predictions_catboost_v2.xlsx"
df_pred.to_excel(out_xlsx, index=False)

print("Файл сохранен:", out_xlsx)

# История прогонов: добавляем новые колонки с версией

history_xlsx = "../data/processed/predictions_history.xlsx"

# Базовые ключи для склейки

key_cols = ["resist_id", "division_title", "operation_id", "date", "id", "month", "year"]

# Текущие метрики для этой версии

# гарантируем, что столбцы CatBoost-метрик существуют и заполнены
if "mape_catboost_ver1" not in df_pred.columns or df_pred["mape_catboost_ver1"].isna().all():
    mape_cat = (df_pred["fact_value"] - df_pred["predict_catboost_ver1"]).abs() / df_pred["fact_value"].abs()
    mask_zero_pred = df_pred["fact_value"] == 0
    mape_cat[mask_zero_pred & (df_pred["predict_catboost_ver1"] == 0)] = 0.0
    mape_cat[mask_zero_pred & (df_pred["predict_catboost_ver1"] != 0)] = 1.0
    mape_cat = mape_cat.replace([float("inf"), float("-inf")], pd.NA)
    df_pred["mape_catboost_ver1"] = mape_cat * 100

if "large_error_share_catboost_ver1" not in df_pred.columns or df_pred["large_error_share_catboost_ver1"].isna().all():
    df_pred["large_error_share_catboost_ver1"] = (df_pred["mape_catboost_ver1"] > 30).fillna(False)


if "square_error_catboost_ver1" not in df_pred.columns or df_pred["square_error_catboost_ver1"].isna().all():
    coef = df_pred["coefficient_fulltime_staff"]
    df_pred["square_error_catboost_ver1"] = (
        (df_pred["fact_value"] * coef) - (df_pred["predict_catboost_ver1"] * coef)
    ).abs() ** 2
    df_pred["square_error_catboost_ver1"] = df_pred["square_error_catboost_ver1"].fillna(0)

base_pred = df_pred.copy()
base_pred = base_pred[key_cols + [
    "predict_catboost_ver1",
    "mape_catboost_ver1",
    "large_error_share_catboost_ver1",
    "square_error_catboost_ver1",
]].copy()

# Переименуем метрики под версионный формат

base_pred = base_pred.rename(columns={
    "predict_catboost_ver1": "predict_catboost_verX",
    "mape_catboost_ver1": "mape_catboost_verX",
    "large_error_share_catboost_ver1": "large_error_share_catboost_verX",
    "square_error_catboost_ver1": "square_error_catboost_verX",
})

df_hist = None
try:
    df_hist = pd.read_excel(history_xlsx)
    if df_hist.empty or not set(key_cols).issubset(df_hist.columns):
        df_hist = None
except (FileNotFoundError, ValueError):
    df_hist = None

if df_hist is None:
    # первая версия в истории — начинаем с ver3
    next_ver = 3
    print("History not found or empty. Start version:", next_ver)
    df_hist = base_pred.rename(columns={
        "predict_catboost_verX": f"predict_catboost_ver{next_ver}",
        "mape_catboost_verX": f"mape_catboost_ver{next_ver}",
        "large_error_share_catboost_verX": f"large_error_share_catboost_ver{next_ver}",
        "square_error_catboost_verX": f"square_error_catboost_ver{next_ver}",
    })
else:
    # определить следующий номер версии по существующим колонкам
    ver_cols = [c for c in df_hist.columns if c.startswith("predict_catboost_ver")]
    if ver_cols:
        max_ver = max(int(c.replace("predict_catboost_ver", "")) for c in ver_cols)
    else:
        max_ver = 2
    next_ver = max_ver + 1
    print("Next version:", next_ver)

    df_new = base_pred.rename(columns={
        "predict_catboost_verX": f"predict_catboost_ver{next_ver}",
        "mape_catboost_verX": f"mape_catboost_ver{next_ver}",
        "large_error_share_catboost_verX": f"large_error_share_catboost_ver{next_ver}",
        "square_error_catboost_verX": f"square_error_catboost_ver{next_ver}",
    })

    # приведение типов ключей перед merge
    for c in ["resist_id", "id", "month", "year"]:
        if c in df_hist.columns:
            df_hist[c] = pd.to_numeric(df_hist[c], errors="coerce").astype("Int64")
        if c in df_new.columns:
            df_new[c] = pd.to_numeric(df_new[c], errors="coerce").astype("Int64")

    # operation_id как строка (в JSON часто строковый ключ)
    if "operation_id" in df_hist.columns:
        df_hist["operation_id"] = df_hist["operation_id"].astype(str)
    if "operation_id" in df_new.columns:
        df_new["operation_id"] = df_new["operation_id"].astype(str)

    if "date" in df_hist.columns:
        df_hist["date"] = pd.to_datetime(df_hist["date"], errors="coerce").dt.date
    if "date" in df_new.columns:
        df_new["date"] = pd.to_datetime(df_new["date"], errors="coerce").dt.date

    # left-merge к истории по ключам
    df_hist = df_hist.merge(df_new, on=key_cols, how="left")

# сохраняем

df_hist.to_excel(history_xlsx, index=False)
print("История сохранена:", history_xlsx)

In [ ]:
# Диагностика распределения fact_value

print(df_model39["fact_value"].describe())
print(df_model39[df_model39["fact_value"] < 5].shape[0] / df_model39.shape[0])  # % мелочи